# 14 · Human-in-the-loop with `interrupt`

**Goal:** by the end you can pause a graph in the middle of a node to ask a person something, resume it later with
their answer, and follow the rules that keep pausing and resuming safe.

## The idea in plain English

Sometimes a graph shouldn't carry on by itself. Before dividing by zero, our calculator should **ask a human** for a
better divisor. **Human-in-the-loop** means a person takes part in the run.

Inside a node, calling **`interrupt(value)`** pauses the graph. LangGraph saves the state with the checkpointer
(lesson 13), stops, and hands `value` (your question) back to the caller in `result["__interrupt__"]`. Nothing is
waiting or blocked. The run is simply saved. Later, maybe hours later, you **resume** by invoking the same thread
with `Command(resume=answer)`. Inside the node, `interrupt(...)` then returns `answer`, and the node finishes.

One surprise to remember: on resume, LangGraph runs the paused node again **from the top**. It doesn't continue from
the middle of the function.

**Analogy:** a bank clerk handling a large transfer. They put the form in the "waiting" tray and phone you to
confirm. When you call back with "yes", they pick the form up and start that step again, this time with your answer.

## Picture

```mermaid
sequenceDiagram
    participant You
    participant Graph
    You->>Graph: invoke({"a": 10, "b": 0}, config)
    Graph-->>You: paused. __interrupt__ = "b is 0. Enter a new divisor:"
    Note over Graph: state saved by the checkpointer
    You->>Graph: invoke(Command(resume=5), config)
    Note over Graph: divide runs again from the top. interrupt() returns 5
    Graph-->>You: {"a": 10, "b": 5, "result": 2.0}
```

## Part 1: Ask for a new divisor

### Step 1: A node that can pause

`divide` prints a line as soon as it starts, so we can see when it runs. If `b` is 0 it calls `interrupt` with a
small dict describing the question. Whatever the human sends back becomes the new `b`.

In [1]:
from typing import Literal, TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt


class DivideState(TypedDict):
    a: float
    b: float
    result: float


def divide(state: DivideState) -> dict:
    print(f"  divide started (b = {state['b']})")
    b = state["b"]
    if b == 0:
        b = interrupt({"question": "b is 0. Enter a new divisor:", "a": state["a"]})
        print(f"  the human answered {b}")
    return {"b": b, "result": state["a"] / b}

### Step 2: Compile with a checkpointer

Pausing means saving, so the graph needs a checkpointer, and every call needs a `thread_id`.

In [2]:
builder = StateGraph(DivideState)
builder.add_node("divide", divide)
builder.add_edge(START, "divide")
builder.add_edge("divide", END)

graph = builder.compile(checkpointer=InMemorySaver())
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	divide(divide)
	__end__([<p>__end__</p>]):::last
	__start__ --> divide;
	divide --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Step 3: Run until it pauses

With `b = 0`, `invoke` returns **early**. There's no `result` yet. Instead there's a special `__interrupt__` key
holding a list of `Interrupt` objects. Each one's `.value` is exactly what the node passed to `interrupt`.

In [3]:
config = {"configurable": {"thread_id": "divide-1"}}

paused = graph.invoke({"a": 10, "b": 0}, config)
print(paused)
print()
print("question for the human:", paused["__interrupt__"][0].value)

  divide started (b = 0)
{'a': 10, 'b': 0, '__interrupt__': [Interrupt(value={'question': 'b is 0. Enter a new divisor:', 'a': 10}, id='da14b426d84a5c3ff7cc3aa622ec57cc', response_schema=None)]}

question for the human: {'question': 'b is 0. Enter a new divisor:', 'a': 10}


The checkpointer agrees: the thread is waiting, and `next` says `divide` is the node that will run when it resumes.

In [4]:
snapshot = graph.get_state(config)
print("next:      ", snapshot.next)
print("interrupts:", [i.value for i in snapshot.interrupts])

next:       ('divide',)
interrupts: [{'question': 'b is 0. Enter a new divisor:', 'a': 10}]


### Step 4: Resume with the human's answer

`Command(resume=5)` (the `Command` from lesson 10) is the answer. We invoke the **same thread**. Watch the printed
lines: `divide started` appears **again**, because the node re-runs from the top. This time `interrupt(...)` returns
`5` instead of pausing.

In [5]:
final = graph.invoke(Command(resume=5), config)
print(final)

  divide started (b = 0)
  the human answered 5
{'a': 10, 'b': 5, 'result': 2.0}


When `b` isn't 0, `interrupt` is never called, and the graph runs straight through like any other.

In [6]:
print(graph.invoke({"a": 10, "b": 4}, {"configurable": {"thread_id": "divide-2"}}))

  divide started (b = 4)
{'a': 10, 'b': 4, 'result': 2.5}


## Part 2: Approve or reject

The human's answer can also choose the route. Here `approve` asks "Divide 10 by 4?". If the answer is `True` it goes
to `divide`; if `False`, to `cancelled`. It returns a `Command(goto=...)` (lesson 10), and the hint
`Command[Literal["divide", "cancelled"]]` puts both routes on the diagram.

In [7]:
class ApprovalState(TypedDict):
    a: float
    b: float
    answer: str


def approve(state: ApprovalState) -> Command[Literal["divide", "cancelled"]]:
    approved = interrupt({"question": f"Divide {state['a']:g} by {state['b']:g}?"})
    return Command(goto="divide" if approved else "cancelled")


def divide_approved(state: ApprovalState) -> dict:
    return {"answer": f"{state['a']:g} / {state['b']:g} = {state['a'] / state['b']:g}"}


def cancelled(state: ApprovalState) -> dict:
    return {"answer": "Cancelled by the human."}

The wiring. `approve` routes with its `Command`, so it needs no edge out of it. `divide` and `cancelled` both lead to `END`.

In [8]:
builder = StateGraph(ApprovalState)
builder.add_node("approve", approve)
builder.add_node("divide", divide_approved)
builder.add_node("cancelled", cancelled)
builder.add_edge(START, "approve")
builder.add_edge("divide", END)
builder.add_edge("cancelled", END)

approval_graph = builder.compile(checkpointer=InMemorySaver())
print(approval_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	approve(approve)
	divide(divide)
	cancelled(cancelled)
	__end__([<p>__end__</p>]):::last
	__start__ --> approve;
	approve -.-> cancelled;
	approve -.-> divide;
	cancelled --> __end__;
	divide --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



We run it twice, on two threads. The human says yes to the first and no to the second.

In [9]:
for thread_id, human_says in [("approve-yes", True), ("approve-no", False)]:
    config = {"configurable": {"thread_id": thread_id}}
    paused = approval_graph.invoke({"a": 10, "b": 4}, config)
    print("asked:   ", paused["__interrupt__"][0].value["question"])
    final = approval_graph.invoke(Command(resume=human_says), config)
    print(f"resume={human_says}:", final["answer"])
    print()

asked:    Divide 10 by 4?
resume=True: 10 / 4 = 2.5

asked:    Divide 10 by 4?
resume=False: Cancelled by the human.



## Part 3: The rules of `interrupt`

Each rule below comes with a small demonstration.

### Rule 1: You need a checkpointer and a `thread_id`

Without a checkpointer, the graph can still pause, but nothing is saved, so there's nothing to resume.

In [10]:
no_memory = StateGraph(DivideState)
no_memory.add_node("divide", divide)
no_memory.add_edge(START, "divide")
no_memory.add_edge("divide", END)
no_memory = no_memory.compile()  # no checkpointer!

print("paused:", "__interrupt__" in no_memory.invoke({"a": 10, "b": 0}))
try:
    no_memory.invoke(Command(resume=5))
except RuntimeError as err:
    print("RuntimeError:", err)

  divide started (b = 0)
paused: True
RuntimeError: Cannot use Command(resume=...) without checkpointer


### Rule 2: The node re-runs from the top when resumed

You already saw `divide started` printed twice in Step 4. Anything a node does **before** `interrupt` happens again
on resume. Printing twice is harmless. Sending an email or charging a card twice is not.

So keep side effects (anything that changes the world outside the graph) **after** the `interrupt`, or in a
separate node that runs after the human has answered.

### Rule 3: Don't wrap `interrupt` in a bare `try/except`

`interrupt` pauses the graph by raising a special exception, `GraphInterrupt`, that LangGraph catches. If your code
catches it first, the pause never happens. Below, `except Exception` swallows it: the graph doesn't pause, the
fallback value `1` is used, and nobody is ever asked.

In [11]:
def divide_swallowing(state: DivideState) -> dict:
    try:
        b = interrupt("b is 0. Enter a new divisor:")
    except Exception as err:  # catches GraphInterrupt too!
        print("  swallowed:", type(err).__name__)
        b = 1
    return {"b": b, "result": state["a"] / b}


builder = StateGraph(DivideState)
builder.add_node("divide", divide_swallowing)
builder.add_edge(START, "divide")
builder.add_edge("divide", END)
swallowing = builder.compile(checkpointer=InMemorySaver())
print(swallowing.invoke({"a": 10, "b": 0}, {"configurable": {"thread_id": "swallow"}}))

  swallowed: GraphInterrupt
{'a': 10, 'b': 1, 'result': 10.0}


If you need a `try/except` around other code, catch only the specific errors you expect (for example
`except ValueError:`), or keep the `interrupt` call outside the `try`.

### Rule 4: Keep several interrupts in a fixed order

A node can call `interrupt` more than once. LangGraph matches resume values to interrupts **by their order** in the
node. Here the node asks for two numbers. Each resume answers the next question, and each time the node re-runs from
the top, replaying the answers it already has.

In [12]:
class TwoNumbersState(TypedDict):
    a: float
    b: float
    result: float


def ask_two_numbers(state: TwoNumbersState) -> dict:
    print("  ask_two_numbers started")
    a = interrupt("First number?")
    b = interrupt("Second number?")
    return {"a": a, "b": b, "result": a + b}


builder = StateGraph(TwoNumbersState)
builder.add_node("ask", ask_two_numbers)
builder.add_edge(START, "ask")
builder.add_edge("ask", END)
two = builder.compile(checkpointer=InMemorySaver())

Three calls on one thread. The first pauses at the first question. The second answers it and pauses at the second question. The third answers that and finishes.

In [13]:
config = {"configurable": {"thread_id": "two-numbers"}}

print("asked:", two.invoke({}, config)["__interrupt__"][0].value)
print("asked:", two.invoke(Command(resume=3), config)["__interrupt__"][0].value)
print("done: ", two.invoke(Command(resume=4), config))

  ask_two_numbers started
asked: First number?
  ask_two_numbers started
asked: Second number?
  ask_two_numbers started
done:  {'a': 3, 'b': 4, 'result': 7}


Because answers are matched by position, don't put an `interrupt` inside an `if` whose answer can change between
runs. Then the second answer could be given to the wrong question.

### Rule 5: Pass JSON-serialisable values

The value you pass to `interrupt` is saved by the checkpointer, so it must be something it can store: numbers,
strings, booleans, `None`, and lists or dicts of those (the kinds of values JSON can hold). A custom object fails.

In [14]:
class Question:  # a custom object, not plain data
    text = "Enter a new divisor:"


def divide_bad_value(state: DivideState) -> dict:
    b = interrupt(Question())
    return {"b": b, "result": state["a"] / b}


builder = StateGraph(DivideState)
builder.add_node("divide", divide_bad_value)
builder.add_edge(START, "divide")
builder.add_edge("divide", END)
bad_value = builder.compile(checkpointer=InMemorySaver())

try:
    bad_value.invoke({"a": 10, "b": 0}, {"configurable": {"thread_id": "bad-value"}})
except TypeError as err:
    print("TypeError:", err)

TypeError: Type is not msgpack serializable: Interrupt


## Part 4: Static breakpoints (a debugging tool)

You can also pause a graph **without** changing any node, by naming nodes at compile time:
`compile(..., interrupt_before=["divide"])`. The graph stops just before `divide` runs. You resume with
`invoke(None, config)`, where `None` means "no new input, just carry on". (There's also `interrupt_after=[...]`.)

This is called a **static breakpoint**. It's handy for checking the state in the middle of a run while debugging.
For asking a human in a real program, use `interrupt()` inside the node.

In [15]:
def plain_divide(state: DivideState) -> dict:
    print("  plain_divide runs")
    return {"result": state["a"] / state["b"]}


builder = StateGraph(DivideState)
builder.add_node("divide", plain_divide)
builder.add_edge(START, "divide")
builder.add_edge("divide", END)
debug_graph = builder.compile(checkpointer=InMemorySaver(), interrupt_before=["divide"])

config = {"configurable": {"thread_id": "breakpoint"}}
print("stopped:", debug_graph.invoke({"a": 9, "b": 3}, config))
print("next:   ", debug_graph.get_state(config).next)
print("resumed:", debug_graph.invoke(None, config))

stopped: {'a': 9, 'b': 3}
next:    ('divide',)
  plain_divide runs
resumed: {'a': 9, 'b': 3, 'result': 3.0}


## What just happened

- **Step 3** ran `divide` with `b = 0`. It returned early with
  `__interrupt__: [Interrupt(value={'question': 'b is 0. Enter a new divisor:', 'a': 10}, ...)]`, and `get_state`
  showed `next: ('divide',)`.
- **Step 4** resumed with `Command(resume=5)`. `divide started` printed again (the node re-ran from the top), then
  `the human answered 5`, and the result was `2.0`.
- **Part 2** asked "Divide 10 by 4?". `resume=True` gave `10 / 4 = 2.5`, and `resume=False` gave
  `Cancelled by the human.`
- **Rule 1** showed that a graph without a checkpointer can pause but can't resume: `RuntimeError`.
- **Rule 3** showed `except Exception` swallowing `GraphInterrupt`: the graph never paused and used `b = 1`.
- **Rule 4** asked two questions in order. Each resume answered the next one, and the result was `3 + 4 = 7`.
- **Rule 5** showed a custom object can't be saved: `TypeError: Type is not msgpack serializable`.
- **Part 4** stopped before `divide` with `interrupt_before`, then carried on with `invoke(None, config)`.

## Common mistakes

1. **Resuming without a checkpointer.**
   `RuntimeError: Cannot use Command(resume=...) without checkpointer`.
   Fix: `compile(checkpointer=InMemorySaver())` (or `SqliteSaver`), and pass a `thread_id`.
2. **Resuming the wrong thread.** `Command(resume=5)` on a thread that isn't paused has nothing to resume. The graph
   starts from an empty state, and the node fails on its first read, for example:
   `KeyError: 'b'`.
   Fix: resume with exactly the same config (`thread_id`) that paused.
3. **Catching the pause.** A `try/except Exception` (or a bare `except:`) around `interrupt` gives **no error**. The
   graph just doesn't pause (Rule 3).
   Fix: keep `interrupt` outside the `try`, or catch only specific errors.

## Try it

1. A person might answer `0` again. Change `divide` so it keeps asking until it gets a divisor that isn't 0. (Hint:
   `while b == 0: b = interrupt(...)`.) Pause with `b = 0`, resume with `0`, then with `4`. What's the result, and how
   many times did `divide started` print?
2. Let the approver change the numbers too. Make `approve` accept an answer like `{"approved": True, "b": 2}` and pass
   the new `b` on with `Command(update={"b": ...}, goto="divide")`. Divide 10 by 4, but resume with
   `{"approved": True, "b": 2}`.

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **Human-in-the-loop**: a run that pauses so a person can answer, approve or edit before it goes on.
- **`interrupt(value)`**: pauses the graph inside a node and sends `value` to the caller. On resume it returns the
  human's answer.
- **`__interrupt__`**: the key in a paused run's result that holds the list of `Interrupt` objects.
- **`Command(resume=...)`**: the input that resumes a paused thread with the human's answer.
- **`GraphInterrupt`**: the special exception `interrupt` uses to pause. Don't catch it.
- **Static breakpoint**: a pause set at compile time with `interrupt_before=[...]` or `interrupt_after=[...]`. Used
  for debugging.